<a href="https://colab.research.google.com/github/spklapjs/Point-6/blob/main/ai_model/notebooks/03_model_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install protobuf==4.25.5 onnx2tf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.onnx
import tensorflow as tf
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#경로설정
BASE_DIR = '/content/drive/MyDrive/point-6/ai_model'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data/processed')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
EXPORT_DIR = os.path.join(BASE_DIR, 'exported_models')
os.makedirs(EXPORT_DIR, exist_ok=True)

In [ ]:
# 2. 전처리된 텐서 데이터 로드 및 미세조정/캘리브레이션용 데이터로더 구성
features_tensor = torch.load(os.path.join(PROCESSED_DIR, 'features.pt'))
labels_tensor = torch.load(os.path.join(PROCESSED_DIR, 'labels.pt')).long()

dataset_size = len(features_tensor)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    TensorDataset(features_tensor, labels_tensor),
    [train_size, val_size, test_size]
)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("대표 데이터셋 공급용 로더 구성 완료")

In [ ]:
# 3. 사용자 정의 CNN-LSTM 하이브리드 모델 구조 선언
class CNNLSTMHybrid(nn.Module):
    def __init__(self, num_classes=6):
        super(CNNLSTMHybrid, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=8, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64)
        )

        self.lstm = nn.LSTM(input_size=64, hidden_size=128, batch_first=True)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        out = self.cnn(x)
        out = out.permute(0, 2, 1)
        out, _ = self.lstm(out)
        out = self.fc(out[:, -1, :])
        return out

model = CNNLSTMHybrid(num_classes=6)
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth')))
print("최고 성능 가중치가 적용된 원본 모델 로드 완료")

In [ ]:
# 4. 채널 푸르닝 적용
for name, module in model.named_modules():
    if isinstance(module, nn.Conv1d):
        prune.ln_structured(module, name='weight', amount=0.93, n=1, dim=0)
        prune.remove(module, 'weight')

print("채널 푸르닝 적용 완료. 정확도 복구를 위한 미세 조정을 시작합니다...")

In [ ]:
# 5. 푸르닝 모델 미세 조정 학습 루프
EPOCHS = 10
LEARNING_RATE = 0.0001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_accuracy = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {accuracy:.2f} 퍼센트")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'pruned_best_model.pth'))

print(f"미세 조정 완료. 최고 검증 정확도: {best_accuracy:.2f} 퍼센트")

model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'pruned_best_model.pth')))
model.eval()

In [ ]:
# 6. ONNX 형식으로 변환 및 내보내기
dummy_input = torch.randn(1, 8, 20)
onnx_model_path = os.path.join(EXPORT_DIR, 'pruned_model.onnx')

torch.onnx.export(
    model,
    dummy_input,
    onnx_model_path,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output']
)
print(f"ONNX 모델 변환 완료: {onnx_model_path}")

In [ ]:
# 7. ONNX를 TensorFlow SavedModel로 변환 후 INT8 선형 양자화 적용
print("ONNX 모델을 TensorFlow SavedModel 형식으로 변환합니다...")
os.system(f"onnx2tf -i {onnx_model_path} -o {EXPORT_DIR}/saved_model")

print("INT8 선형 양자화를 적용하여 TFLite 모델로 변환합니다...")
converter = tf.lite.TFLiteConverter.from_saved_model(f"{EXPORT_DIR}/saved_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]

In [ ]:
# 실제 센서 데이터를 활용하는 대표 데이터셋 생성 함수
def representative_dataset():
    # 학습 데이터 로더에서 실제 센서 배치를 가져와 캘리브레이션 수행
    for inputs, _ in train_loader:
        # 파이토치 텐서를 텐서플로우가 요구하는 32비트 부동소수점 형식으로 캐스팅
        tf_input = tf.cast(inputs.numpy(), tf.float32)
        yield [tf_input]

In [ ]:
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

tflite_model_path = os.path.join(EXPORT_DIR, 'motion_model_quantized.tflite')
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_quant_model)

print(f"INT8 양자화 TFLite 모델 변환 완료: {tflite_model_path}")